# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb, os
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet') LIMIT 5").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

1. The contract (5 answers)

One row = one pseudonymized content item (content_hash_id) that was active in fact_content_daily_performance during month=2026-03
Table(s) = dim_content.parquet (primary), fact_content_daily_performance/month=2026-03/ (to scope "active" + validate)
Time window = month=2026-03
Predict/rank = no true label; you're producing a cluster assignment (archetype ID) per content item — unsupervised
Excluded = all performance columns (gsc_*, ga4_*, sessions_*, ai_*) kept out of clustering features, because including them would make clusters split by "performs well vs. poorly" instead of by structure

2. Prove three facts with three queries

Grain check → Query A (groups by content_hash_id, checks for duplicates)
Row count + date span → Query B (on the month=2026-03 fact partition)
Availability with IS TRUE → Query C (is_published IS TRUE, is_deleted IS NOT TRUE)

3. Five features
word_count, char_count, keyword_token_count, category_count, content_type/main_intent — all knowable at authoring time, before any traffic exists.

4. The trap
Add avg_clicks (a performance metric) as a clustering feature → clusters separate almost perfectly along performance instead of structure → delete it → keep the honest, messier structural clustering as your real result.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row is one pseudonymized content item (content_hash_id) from dim_content, restricted to items that had at least one reporting row in fact_content_daily_performance during month=2026-03. This is a content-level grain, not a content-day grain — appropriate for archetype clustering, which groups content by structural properties rather than daily behavior.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: content_type, main_intent, word_count, char_count, keyword_token_count, category_count — all static structural properties set at authoring time.
Label: none — unsupervised task; cluster ID is the output, not a trained-against label.
Context: client_hash_id (join/scoping only), search_volume, cpc, competition_level, backlinks (SEO market context, useful for interpreting clusters afterward, not used as clustering input).
Excluded: all gsc_*/ga4_*/sessions_*/ai_* performance columns (would leak outcome into structure-based clusters — see Part 4); content_hash_id/keyword_hash_id/url_hash_id (identifiers, not signal); last_optimized_date/optimization_eligible_date (workflow metadata, not structure); is_deleted (filter, not feature).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
rel = "hf://datasets/FlyRank/internship-warehouse"

# Query A — grain check: one row really is one content item
con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n
    FROM read_parquet('{rel}/dim_content.parquet')
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").show()
# Query B — active-content row count + date span for month=2026-03
con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS active_content_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS n_days
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").show()
# Query C — availability with IS TRUE (published, not deleted)
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNTIF(is_published IS TRUE) AS published_rows,
        COUNTIF(is_deleted IS TRUE) AS deleted_rows,
        ROUND(COUNTIF(is_published IS TRUE AND is_deleted IS NOT TRUE) * 1.0 / COUNT(*), 3) AS pct_usable
    FROM read_parquet('{rel}/dim_content.parquet')
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────┐
│ content_hash_id │   n   │
│     varchar     │ int64 │
└─────────────────┴───────┘
          0 rows         



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬────────────┬────────────┬────────┐
│ active_content_count │  min_date  │  max_date  │ n_days │
│        int64         │    date    │    date    │ int64  │
├──────────────────────┼────────────┼────────────┼────────┤
│               331437 │ 2026-03-01 │ 2026-03-31 │     31 │
└──────────────────────┴────────────┴────────────┴────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell you about content created or deleted outside its own history, or about clients whose GSC/GA4 access started partway through the panel — dim_clients.gsc_data_start / ga4_data_start show per-client history depth is uneven, so early months may be GSC-only or GA4-only for some clients. Structural clustering on dim_content also can't capture how a content item's structure changed over time — content_updated_date exists, but this snapshot doesn't preserve prior versions, so a heavily-revised page and a never-touched page look identical if their current structural stats match.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.